# 00 · Setup Base — Proyecto de Restauración de Audio (TFG)

Notebook base del proyecto. Aquí se configura la persistencia en Drive, la caché de Hugging Face,
la estructura de carpetas, las utilidades comunes de audio, el módulo de métricas y los baselines
clásicos no-IA.

**Este notebook NO carga ningún modelo de deep learning.** Cada modelo (MP-SENet, DeepFilterNet2/3,
AudioSR, VoiceFixer v2, HTDemucs v4, ClearVoice/MossFormer2) tiene su propio notebook de prueba
aislado, ya que cada uno se instala y se usa de forma distinta (paquete pip propio, repo de GitHub
clonado, o modelo de `transformers`). Este notebook solo prepara el terreno común que todos van a
reutilizar.

**Flujo de trabajo:**
1. Ejecutar este notebook una vez por sesión de Colab (monta Drive, prepara carpetas, define utils).
2. Guardar las funciones de este notebook como módulo importable en Drive (`utils/`), para poder
   `import` las mismas funciones desde los notebooks de cada modelo sin copiar/pegar código.
3. En cada notebook de modelo: instalar solo las dependencias de ESE modelo, cargarlo, probar
   inferencia sobre un audio de `audio_samples/`, guardar resultado en `outputs/`, calcular métricas.
4. Al final, un notebook de integración con Gradio que importa todas las funciones de inferencia
   ya probadas y las expone en la interfaz.


## 1. Montaje de Google Drive

In [2]:
# Montamos Google Drive para persistencia de modelos, audios y resultados entre sesiones.
# Sin esto, cada vez que Colab reinicie el entorno (por inactividad o desconexión) se perdería
# todo lo descargado, que en el caso de estos modelos puede suponer varios GB por sesión.

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 2. Estructura de carpetas del proyecto y caché de Hugging Face

Configuramos `HF_HOME` y `HF_HUB_CACHE` para que cualquier modelo descargado desde Hugging Face
(vía `transformers`, `huggingface_hub`, o paquetes que usen el hub por debajo) se guarde directamente
en Drive en lugar del disco efímero de Colab (`/content`, que se borra al reiniciar).

**Importante:** estas variables de entorno deben configurarse ANTES de importar `transformers` o
`huggingface_hub` en cualquier notebook (incluidos los de cada modelo), o no tendrán efecto.


In [3]:
import os

# --- Ruta raíz del proyecto en Drive ---
PROJECT_ROOT = '/content/drive/MyDrive/Proyecto_Audio'

# --- Subcarpetas del proyecto ---
PATHS = {
    'audio_samples': f'{PROJECT_ROOT}/audio_samples',   # audios de entrada / prueba (incluye el audio real del tutor)
    'outputs':       f'{PROJECT_ROOT}/outputs',          # resultados generados por cada modelo, organizados por subcarpeta
    'notebooks':     f'{PROJECT_ROOT}/notebooks',        # notebooks del proyecto (copia de respaldo)
    'utils':         f'{PROJECT_ROOT}/utils',            # módulos .py compartidos (audio_utils.py, metrics.py, baselines.py)
    'checkpoints':   f'{PROJECT_ROOT}/checkpoints',      # checkpoints de modelos que NO se gestionan vía HF_HUB_CACHE (ej. repos de GitHub)
    'cache':         f'{PROJECT_ROOT}/cache',            # caché de Hugging Face (HF_HOME / HF_HUB_CACHE)
}

# --- Creamos las carpetas si no existen (idempotente: no falla si ya existen) ---
for nombre, ruta in PATHS.items():
    os.makedirs(ruta, exist_ok=True)
    print(f'✓ {nombre:<12} -> {ruta}')

# --- Redirigimos la caché de Hugging Face a Drive ---
# HF_HOME afecta a la caché general (modelos, datasets, tokenizers).
# HF_HUB_CACHE es más específico solo para el hub; fijamos ambas por compatibilidad entre versiones.
os.environ['HF_HOME'] = PATHS['cache']
os.environ['HF_HUB_CACHE'] = PATHS['cache']

print(f'\nHF_HOME configurado en: {os.environ["HF_HOME"]}')
print('A partir de ahora, cualquier modelo descargado vía transformers/huggingface_hub')
print('se guardará en Drive y no se volverá a descargar en próximas sesiones.')


✓ audio_samples -> /content/drive/MyDrive/Proyecto_Audio/audio_samples
✓ outputs      -> /content/drive/MyDrive/Proyecto_Audio/outputs
✓ notebooks    -> /content/drive/MyDrive/Proyecto_Audio/notebooks
✓ utils        -> /content/drive/MyDrive/Proyecto_Audio/utils
✓ checkpoints  -> /content/drive/MyDrive/Proyecto_Audio/checkpoints
✓ cache        -> /content/drive/MyDrive/Proyecto_Audio/cache

HF_HOME configurado en: /content/drive/MyDrive/Proyecto_Audio/cache
A partir de ahora, cualquier modelo descargado vía transformers/huggingface_hub
se guardará en Drive y no se volverá a descargar en próximas sesiones.


## 3. Dependencias comunes

Estas son las librerías que se van a usar en prácticamente todos los notebooks (I/O de audio,
resampling, métricas, gráficas). Las dependencias específicas de cada modelo (ej. `deepfilternet`,
`clearvoice`, el repo de MP-SENet, etc.) se instalan en su propio notebook, no aquí.


In [4]:
!pip install -q librosa soundfile scipy pesq pystoi speechmos matplotlib


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 78.5 MB/s eta 0:00:00


## 4. Utilidades comunes de audio (I/O)

Funciones básicas de carga, guardado, resampling y normalización mono/estéreo. Cada modelo espera
el audio en un formato distinto (sample rate, mono/estéreo), así que estas funciones son las que
adaptan tu audio de entrada a lo que pida cada uno.


In [15]:
%%writefile /content/drive/MyDrive/Proyecto_Audio/utils/audio_utils_funcionescomunes.py
import librosa
import soundfile as sf
import numpy as np


def cargar_audio(ruta, sr_objetivo=None, forzar_mono=True):
    """
    Carga un archivo de audio desde disco.

    Args:
        ruta (str): ruta al archivo de audio (.wav, .mp3, etc.)
        sr_objetivo (int, opcional): si se especifica, resamplea al cargar.
                                     Si es None, se mantiene el sample rate original.
        forzar_mono (bool): si True, hace downmix a mono (promedio de canales).

    Returns:
        audio (np.ndarray): señal de audio como array 1D (mono) o 2D (canales, muestras).
        sr (int): sample rate real del audio devuelto.
    """
    audio, sr = librosa.load(ruta, sr=sr_objetivo, mono=forzar_mono)
    return audio, sr


def guardar_audio(ruta_salida, audio, sr):
    """
    Guarda una señal de audio en disco como .wav.

    Args:
        ruta_salida (str): ruta de destino, debe terminar en .wav
        audio (np.ndarray): señal de audio (mono o estéreo)
        sr (int): sample rate de la señal
    """
    sf.write(ruta_salida, audio, sr)
    print(f'Audio guardado en: {ruta_salida}')


def resamplear(audio, sr_origen, sr_destino):
    """
    Resamplea una señal de audio de sr_origen a sr_destino.
    Necesario porque cada modelo trabaja a una frecuencia distinta
    (ej. DeepFilterNet a 48kHz, MP-SENet a 16kHz, AudioSR variable).
    """
    if sr_origen == sr_destino:
        return audio
    return librosa.resample(audio, orig_sr=sr_origen, target_sr=sr_destino)


def normalizar_pico(audio, pico_objetivo=0.95):
    """
    Normaliza el audio para que su valor de pico absoluto sea pico_objetivo.
    Útil antes de aplicar clipping sintético o antes de pasar por modelos
    sensibles al nivel de entrada.
    """
    pico_actual = np.max(np.abs(audio))
    if pico_actual == 0:
        return audio
    return audio * (pico_objetivo / pico_actual)


Writing /content/drive/MyDrive/Proyecto_Audio/utils/audio_utils_funcionescomunes.py


## 5. Gestión de memoria GPU

Para poder probar varios modelos en la misma sesión de Colab sin quedarnos sin VRAM (Out-Of-Memory),
es importante liberar la memoria de la GPU después de usar cada modelo. Esta función se llama al
final de cada notebook de modelo, después de terminar la inferencia.


In [16]:
%%writefile /content/drive/MyDrive/Proyecto_Audio/utils/audio_utils_memoria_GPU.py
import gc
import torch


def liberar_memoria_gpu(modelo=None):
    """
    Libera la memoria GPU ocupada por un modelo de PyTorch.

    Args:
        modelo: objeto del modelo a eliminar (opcional). Si se pasa, se borra la referencia
                explícitamente antes de vaciar la caché de CUDA.
    """
    if modelo is not None:
        del modelo
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f'Memoria GPU liberada. Uso actual: {torch.cuda.memory_allocated() / 1e9:.2f} GB')
    else:
        print('No hay GPU disponible en esta sesión (Entorno de ejecución > Cambiar tipo de entorno > GPU).')


Writing /content/drive/MyDrive/Proyecto_Audio/utils/audio_utils_memoria_GPU.py


## 6. Métricas con referencia (PESQ, STOI, SI-SDR, LSD)

Estas métricas requieren un audio limpio de referencia (ground-truth), por lo que se usan sobre todo
para replicar comparativas del bloque bibliográfico o si en algún momento generas tus propios pares
degradado/limpio. **No aplican al audio real del tutor**, ya que no tienes una versión limpia de
referencia de ese audio (para eso están las métricas no-intrusivas de la siguiente sección).


In [17]:
%%writefile /content/drive/MyDrive/Proyecto_Audio/utils/audio_utils_metricas_ref.py
from pesq import pesq
from pystoi import stoi


def calcular_pesq(audio_referencia, audio_procesado, sr):
    """
    PESQ (Perceptual Evaluation of Speech Quality). Rango aproximado: -0.5 a 4.5 (más alto = mejor).
    Requiere sr de 8000 o 16000 Hz (la librería `pesq` reamplea internamente si hace falta, pero
    es recomendable pasar el audio ya a 16kHz para evitar errores).
    """
    modo = 'wb' if sr >= 16000 else 'nb'  # wideband si sr>=16kHz, narrowband si no
    return pesq(sr, audio_referencia, audio_procesado, modo)


def calcular_stoi(audio_referencia, audio_procesado, sr):
    """
    STOI (Short-Time Objective Intelligibility). Rango 0 a 1 (más alto = más inteligible).
    """
    return stoi(audio_referencia, audio_procesado, sr, extended=False)


def calcular_si_sdr(audio_referencia, audio_procesado):
    """
    SI-SDR (Scale-Invariant Signal-to-Distortion Ratio), en dB (más alto = mejor).
    Implementación manual (no requiere librería adicional).
    """
    referencia = audio_referencia - np.mean(audio_referencia)
    procesado = audio_procesado - np.mean(audio_procesado)

    # Proyección escalada de la señal procesada sobre la referencia
    alpha = np.dot(procesado, referencia) / (np.dot(referencia, referencia) + 1e-8)
    proyeccion = alpha * referencia
    ruido = procesado - proyeccion

    return 10 * np.log10((np.sum(proyeccion ** 2) + 1e-8) / (np.sum(ruido ** 2) + 1e-8))


def calcular_lsd(audio_referencia, audio_procesado, n_fft=1024):
    """
    LSD (Log-Spectral Distance), en dB (más bajo = mejor). Muy usada en papers de BWE/super-resolución
    para medir diferencias espectrales, especialmente en las frecuencias altas reconstruidas.
    """
    ref_stft = np.abs(librosa.stft(audio_referencia, n_fft=n_fft))
    proc_stft = np.abs(librosa.stft(audio_procesado, n_fft=n_fft))

    min_frames = min(ref_stft.shape[1], proc_stft.shape[1])
    ref_stft, proc_stft = ref_stft[:, :min_frames], proc_stft[:, :min_frames]

    log_ref = np.log10(np.maximum(ref_stft, 1e-8) ** 2)
    log_proc = np.log10(np.maximum(proc_stft, 1e-8) ** 2)

    distancia_por_frame = np.sqrt(np.mean((log_ref - log_proc) ** 2, axis=0))
    return np.mean(distancia_por_frame)


Writing /content/drive/MyDrive/Proyecto_Audio/utils/audio_utils_metricas_ref.py


## 7. Métricas no-intrusivas (DNSMOS, NISQA)

Estas métricas NO requieren audio de referencia limpio: estiman la calidad percibida directamente
sobre el audio procesado. **Son las que usarás para el bloque empírico con el audio real de tu
tutor**, ya que ahí no existe una versión limpia con la que comparar.


In [18]:
%%writefile /content/drive/MyDrive/Proyecto_Audio/utils/audio_utils_metricas_no_intrusivas.py
from speechmos import dnsmos

def calcular_dnsmos(audio, sr):
    """
    DNSMOS (Deep Noise Suppression MOS). Devuelve una estimación de MOS (Mean Opinion Score)
    sin necesidad de referencia limpia. Ideal para el audio real del tutor.

    Devuelve un diccionario con submétricas: SIG (calidad de la señal de voz),
    BAK (calidad del ruido de fondo), OVRL (calidad general).
    """
    resultado = dnsmos.run(audio, sr=sr)
    return resultado


# NISQA requiere clonar su repo oficial (no está empaquetado en pip de forma estable):
# !git clone https://github.com/gabrielmittag/NISQA.git
# Se deja como paso pendiente para el notebook de métricas empíricas, ya que necesita
# descargar sus propios pesos preentrenados (no vía HF_HOME).


Writing /content/drive/MyDrive/Proyecto_Audio/utils/audio_utils_metricas_no_intrusivas.py


## 8. Baselines clásicos no-IA

Un baseline clásico (sin deep learning) por categoría, para la comparativa IA vs. no-IA que planteas
en tu metodología. Son deliberadamente simples: el objetivo no es competir con los modelos de IA,
sino tener un punto de referencia clásico bien entendido y reproducible.


In [19]:
%%writefile /content/drive/MyDrive/Proyecto_Audio/utils/audio_utils_baselines_clasicos.py
import scipy.signal as signal


def baseline_denoising_spectral_gating(audio, sr, umbral_db=-20):
    """
    Baseline de denoising: 'spectral gating' clásico. Estima el ruido de fondo a partir de los
    frames más silenciosos y resta ese perfil espectral del resto de la señal.
    """
    stft = librosa.stft(audio)
    magnitud, fase = np.abs(stft), np.angle(stft)

    # Estimamos el perfil de ruido como el percentil 10 de energía por banda de frecuencia
    perfil_ruido = np.percentile(magnitud, 10, axis=1, keepdims=True)

    umbral = perfil_ruido * (10 ** (umbral_db / 20))
    magnitud_limpia = np.where(magnitud > umbral, magnitud - perfil_ruido, 0.0)
    magnitud_limpia = np.maximum(magnitud_limpia, 0.0)

    stft_limpio = magnitud_limpia * np.exp(1j * fase)
    return librosa.istft(stft_limpio, length=len(audio))


def baseline_dereverb_filtro_paso_alto(audio, sr, frecuencia_corte=100):
    """
    Baseline de dereverberation: filtro paso-alto simple. No elimina reverberación de forma
    sofisticada, pero atenúa la acumulación de energía de baja frecuencia típica de la cola
    reverberante en salas. Sirve como referencia clásica mínima.
    """
    sos = signal.butter(4, frecuencia_corte, btype='high', fs=sr, output='sos')
    return signal.sosfilt(sos, audio)


def baseline_bwe_interpolacion_spline(audio, sr_origen, sr_destino):
    """
    Baseline de super-resolución/BWE: upsampling clásico por interpolación spline cúbica.
    No genera contenido de alta frecuencia real, solo interpola las muestras existentes.
    """
    return librosa.resample(audio, orig_sr=sr_origen, target_sr=sr_destino, res_type='fft')


def baseline_declipping_interpolacion_cubica(audio, umbral=0.99):
    """
    Baseline de de-clipping: detecta las muestras saturadas (por encima del umbral) y las
    reconstruye por interpolación cúbica con las muestras vecinas no saturadas.
    """
    audio_reparado = audio.copy()
    indices_clipeados = np.where(np.abs(audio) >= umbral)[0]

    if len(indices_clipeados) == 0:
        return audio_reparado  # no hay clipping que reparar

    indices_validos = np.where(np.abs(audio) < umbral)[0]
    if len(indices_validos) < 4:
        return audio_reparado  # no hay suficientes puntos para interpolar

    from scipy.interpolate import CubicSpline
    spline = CubicSpline(indices_validos, audio[indices_validos])
    audio_reparado[indices_clipeados] = spline(indices_clipeados)
    return audio_reparado


Writing /content/drive/MyDrive/Proyecto_Audio/utils/audio_utils_baselines_clasicos.py


## 9. Próximos pasos

1. Guardar las celdas de utilidades (secciones 4, 5, 6, 7, 8) como módulos `.py` en
   `utils/` dentro de Drive, para poder hacer `import` desde los notebooks de cada modelo
   en lugar de copiar/pegar código.
2. Crear un notebook aislado por modelo (`01_DeepFilterNet.ipynb`, `02_MPSENet.ipynb`, etc.),
   cada uno con sus propias dependencias, que importe estas utilidades comunes.
3. Cuando los 6 modelos + baselines estén probados individualmente, crear el notebook de
   integración con Gradio que importe las funciones de inferencia ya validadas.


In [14]:
!ls -la /content/drive/MyDrive/Proyecto_Audio/

total 24
drwx------ 2 root root 4096 Jul 10 07:15 audio_samples
drwx------ 2 root root 4096 Jul 10 07:15 cache
drwx------ 2 root root 4096 Jul 10 07:15 checkpoints
drwx------ 2 root root 4096 Jul 10 07:15 notebooks
drwx------ 2 root root 4096 Jul 10 07:15 outputs
drwx------ 2 root root 4096 Jul 10 07:34 utils
